### load packages, data

In [131]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(42)

In [132]:
# # shakespeare dataset

# !wget https://raw.githubusercontent.com/AviSoori1x/makeMoE/main/input.txt

### define expert model

In [133]:
class Expert(nn.Module):
    """linear layer followed by relu activation"""

    def __init__(self, n_embd):
        # n_embd denotes embedding dimension
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # expansion
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), # contraction
            nn.Dropout(0.1), # 10% dropout, tune later
            # dropout=0.1
        )
    
    def forward(self, x):
        return self.net(x)

### implement router

- input matrix: count of tokens x embedding dimension 
- routing matrix: embedding dimension x number of experts
- input matrix X routing matrix = expert selector matrix
- expert selector matrix: tokens x number of experts

- routger determines which expert network receives the output for each token after/from multi-head attention

#### top k-load balancing after

In [134]:
# gating
num_experts = 3
top_k = 2 # top k experts to select for each token
n_embed = 8

# ex multi-head attention output for example
# consider n_embed=32, context length=4, ...

mh_output = torch.randn(1, 4, n_embed) # batch_size, context_length, embedding_dim

topkgate_linear = nn.Linear(n_embed, num_experts) # router

logits = topkgate_linear(mh_output) # (1, 4, 3)

print(logits) # denotes expert selector

# each row corresponds to one token
# every col corresponds to one expert

tensor([[[ 0.0238, -0.2771, -0.5070],
         [-0.5727, -0.9081,  0.1839],
         [ 0.8137,  0.1781,  1.5661],
         [ 0.6523,  0.4525,  0.0062]]], grad_fn=<ViewBackward0>)


### top_k load balancing

In [135]:
top_k_logits, top_k_indices = logits.topk(top_k, dim=-1) # (1, 4, 2) obtain top-k experts for each token
top_k_logits, top_k_indices

(tensor([[[ 0.0238, -0.2771],
          [ 0.1839, -0.5727],
          [ 1.5661,  0.8137],
          [ 0.6523,  0.4525]]], grad_fn=<TopkBackward0>),
 tensor([[[0, 1],
          [2, 0],
          [2, 0],
          [0, 1]]]))

### -inf and apply softmax

In [136]:
zeros = torch.full_like(logits, float('-inf')) # (1, 4, 3) create a tensor of -inf values to mask out non-top-k experts
sparse_logits = zeros.scatter(-1, top_k_indices, top_k_logits) # (1, 4, 3) scatter the top-k logits into the zeros tensor
sparse_logits

tensor([[[ 0.0238, -0.2771,    -inf],
         [-0.5727,    -inf,  0.1839],
         [ 0.8137,    -inf,  1.5661],
         [ 0.6523,  0.4525,    -inf]]], grad_fn=<ScatterBackward0>)

In [137]:
gating_output = F.softmax(sparse_logits, dim=-1) # (1, 4, 3) softmax to get gating probabilities
gating_output

tensor([[[0.5747, 0.4253, 0.0000],
         [0.3194, 0.0000, 0.6806],
         [0.3203, 0.0000, 0.6797],
         [0.5498, 0.4502, 0.0000]]], grad_fn=<SoftmaxBackward0>)

In [138]:
class TopkRouter(nn.Module):
    def __init__(self, n_embed, num_experts, top_k):
        super(TopkRouter, self).__init__() # initialize the router with a linear layer to compute logits for expert selection
        self.top_k = top_k
        self.linear = nn.Linear(n_embed, num_experts)
    
    def forward(self, mh_output): # mh_output is the output from multi-head attention, shape (batch_size, context_length, embedding_dim)
        logits = self.linear(mh_output) # compute logits for expert selection
        top_k_logits, indices = logits.topk(self.top_k, dim=-1) # get top-k experts for each token
        zeros = torch.full_like(logits, float('-inf')) # create a tensor of -inf values to mask out non-top-k experts
        sparse_logits = zeros.scatter(-1, indices, top_k_logits) # scatter the top-k logits into the zeros tensor
        gating_output = F.softmax(sparse_logits, dim=-1) # softmax to get gating probabilities
        return gating_output, indices # result is expert selector weight matrix

In [139]:
# testing
num_experts = 3
top_k = 2
n_embed = 8 # embedding dimension

mh_output = torch.randn(1, 4, n_embed) # example multi-head attention output
top_k_gate = TopkRouter(n_embed, num_experts, top_k) # initialize the router
gating_output, indices = top_k_gate(mh_output) # get gating output and selected
gating_output.shape, gating_output, indices

(torch.Size([1, 4, 3]),
 tensor([[[0.6177, 0.0000, 0.3823],
          [0.6445, 0.3555, 0.0000],
          [0.0000, 0.3600, 0.6400],
          [0.5666, 0.4334, 0.0000]]], grad_fn=<SoftmaxBackward0>),
 tensor([[[0, 2],
          [0, 1],
          [2, 1],
          [0, 1]]]))

In [140]:
# create class for NoisyTopK Routing

# adding noise to introduce some form of choas to ensure all experts get trained and to prevent expert collapse

# added to expert selector matrix

class NoisyTopkRouter(nn.Module):
    def __init__(self, n_embed, num_experts, top_k):
        super(NoisyTopkRouter, self).__init__()
        self.top_k = top_k
        # layer for router logits
        self.topkroute_linear = nn.Linear(n_embed, num_experts)
        self.noise_linear = nn.Linear(n_embed, num_experts)

    def forward(self, mh_output):
        # mh_output is the output tensor from multihead self attention block
        logits = self.topkroute_linear(mh_output)

        # noise logits, 
        noise_logits = self.noise_linear(mh_output) 

        # adding scaled unit gaussian noise to the logits
        noise = torch.randn_like(logits)*F.softplus(noise_logits)
        noisy_logits = logits + noise

        top_k_logits, indices = noisy_logits.topk(self.top_k, dim=-1)
        zeros = torch.full_like(logits, float('-inf'))
        sparse_logits = zeros.scatter(-1, indices, top_k_logits)
        router_output = F.softmax(sparse_logits, dim=-1)
        return router_output, indices

In [141]:
# testing
num_experts = 3
top_k = 2
n_embed = 8 # embedding dimension

mh_output = torch.randn(1, 4, n_embed) # example multi-head attention output
noisy_top_k_gate = NoisyTopkRouter(n_embed, num_experts, top_k) # initialize the router
gating_output, indices = noisy_top_k_gate(mh_output) # get gating output and selected
gating_output.shape, gating_output, indices

(torch.Size([1, 4, 3]),
 tensor([[[0.6890, 0.0000, 0.3110],
          [0.0000, 0.2910, 0.7090],
          [0.0000, 0.3835, 0.6165],
          [0.0000, 0.4379, 0.5621]]], grad_fn=<SoftmaxBackward0>),
 tensor([[[0, 2],
          [2, 1],
          [2, 1],
          [2, 1]]]))

In [142]:
# create the sparse MoE model
class SparseMoE(nn.Module):
    def __init__(self, n_embed, num_experts, top_k):
        super(SparseMoE, self).__init__()
        self.top_k = top_k
        self.router = NoisyTopkRouter(n_embed, num_experts, top_k)
        self.experts = nn.ModuleList([Expert(n_embed) for _ in range(num_experts)]) # create a list of expert modules
    
    def forward(self, x):
        # x is the input tensor to the MoE layer, shape (batch_size, context_length, embedding_dim)
        gating_output, indices = self.router(x) # get gating output and selected expert indices
        final_output = torch.zeros_like(x) # initialize a tensor to hold the final output, shape (batch_size, context_length, embedding_dim)

        # reshape inputs for batch processing
        flat_x = x.view(-1, x.size(-1)) # shape (batch_size * context_length, embedding_dim)
        flat_gating_output = gating_output.view(-1, gating_output.size(-1)) # shape (batch_size * context_length, num_experts)

        # process each expert in parallel
        # ADVANTAGE: we can process all tokens for a given expert in one forward pass, which is more efficient than processing each token separately
        for i, expert in enumerate(self.experts):
            # create mask for the inputs where the cur expert is in top_k
            expert_mask = (indices == i).any(dim=-1) # shape (batch_size, context_length)
            flat_mask = expert_mask.view(-1) # shape (batch_size * context_length)

            if flat_mask.any():
                # select the inputs for the current expert
                expert_input = flat_x[flat_mask] # shape (num_tokens_for_expert_i, embedding_dim)
                expert_output = expert(expert_input) # shape (num_tokens_for_expert_i, embedding_dim)

                # extract and apply gating scores
                expert_gating_weights = flat_gating_output[flat_mask, i] # shape (num_tokens_for_expert_i,)

                # update final output additiviely by indexing and adding
                final_output.view(-1, final_output.size(-1))[flat_mask] += expert_gating_weights.unsqueeze(-1) * expert_output

        return final_output

In [143]:
# test

import torch
import torch.nn as nn

num_experts = 3
top_k = 2
n_embed = 8
dropout = 0.1

mh_output = torch.randn(1, 4, n_embed) # batch_size=1, context_length=4, embedding_dim=8
sparse_moe = SparseMoE(n_embed, num_experts, top_k) # initialize the sparse MoE model
final_output = sparse_moe(mh_output) # get the final output from the MoE layer
print("shape of final output:", final_output.shape) # should be (1, 4, 8)
print("final output:", final_output)

shape of final output: torch.Size([1, 4, 8])
final output: tensor([[[ 0.0859,  0.0951,  0.2380, -0.3944,  0.2567, -0.1257,  0.1447,
          -0.0695],
         [-0.1468,  0.1766, -0.0747, -0.0626,  0.3956,  0.0657, -0.1034,
          -0.1076],
         [ 0.0343,  0.0599,  0.0000, -0.2399,  0.1180,  0.0310, -0.1273,
          -0.0255],
         [ 0.0000,  0.2572, -0.1196, -0.2590,  0.3636,  0.0703,  0.0114,
           0.1222]]], grad_fn=<CopySlices>)


In [144]:
# putting it all together

# expert module
class Expert(nn.Module):
    """linear layer followed by relu activation"""

    def __init__(self, n_embd):
        # n_embd denotes embedding dimension
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), # expansion
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd), # contraction
            nn.Dropout(dropout), # 10% dropout, tune later
            # dropout=0.1
        )
    
    def forward(self, x):
        return self.net(x)
    
# changing the above to accomodate noisy top-k gating
# gives expert selector weight matrix w/ noise
class NoisyTopkRouter(nn.Module):
    def __init__(self, n_embed, num_experts, top_k):
        super(NoisyTopkRouter, self).__init__()
        self.top_k = top_k
        # layer for router logits
        self.topkroute_linear = nn.Linear(n_embed, num_experts)
        self.noise_linear = nn.Linear(n_embed, num_experts)

    def forward(self, mh_output):
        # mh_output is the output tensor from multihead self attention block
        logits = self.topkroute_linear(mh_output)

        # noise logits, 
        noise_logits = self.noise_linear(mh_output) 

        # adding scaled unit gaussian noise to the logits
        noise = torch.randn_like(logits)*F.softplus(noise_logits)
        noisy_logits = logits + noise

        top_k_logits, indices = noisy_logits.topk(self.top_k, dim=-1)
        zeros = torch.full_like(logits, float('-inf'))
        sparse_logits = zeros.scatter(-1, indices, top_k_logits)
        router_output = F.softmax(sparse_logits, dim=-1)
        return router_output, indices
    
# sparse MoE model
class SparseMoE(nn.Module):
    def __init__(self, n_embed, num_experts, top_k):
        super(SparseMoE, self).__init__()
        self.top_k = top_k
        self.router = NoisyTopkRouter(n_embed, num_experts, top_k)
        self.experts = nn.ModuleList([Expert(n_embed) for _ in range(num_experts)]) # create a list of expert modules
    
    def forward(self, x):
        # x is the input tensor to the MoE layer, shape (batch_size, context_length, embedding_dim)
        gating_output, indices = self.router(x) # get gating output and selected expert indices
        final_output = torch.zeros_like(x) # initialize a tensor to hold the final output, shape (batch_size, context_length, embedding_dim)

        # reshape inputs for batch processing
        flat_x = x.view(-1, x.size(-1)) # shape (batch_size * context_length, embedding_dim)
        flat_gating_output = gating_output.view(-1, gating_output.size(-1)) # shape (batch_size * context_length, num_experts)

        # process each expert in parallel
        # ADVANTAGE: we can process all tokens for a given expert in one forward pass, which is more efficient than processing each token separately
        for i, expert in enumerate(self.experts):
            # create mask for the inputs where the cur expert is in top_k
            expert_mask = (indices == i).any(dim=-1) # shape (batch_size, context_length)
            flat_mask = expert_mask.view(-1) # shape (batch_size * context_length)

            if flat_mask.any():
                # select the inputs for the current expert
                expert_input = flat_x[flat_mask] # shape (num_tokens_for_expert_i, embedding_dim)
                expert_output = expert(expert_input) # shape (num_tokens_for_expert_i, embedding_dim)

                # extract and apply gating scores
                expert_gating_weights = flat_gating_output[flat_mask, i] # shape (num_tokens_for_expert_i,)

                # update final output additiviely by indexing and adding
                final_output.view(-1, final_output.size(-1))[flat_mask] += expert_gating_weights.unsqueeze(-1) * expert_output

        return final_output

In [145]:
# code entire transformer black: p1 multi-head attention

class Head(nn.Module):
    """one head of self attention"""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias = False)
        self.query = nn.Linear(n_embed, head_size, bias = False)
        self.value = nn.Linear(n_embed, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        B, T, C = x.shape # batch size, context length, embedding dimension
        k = self.key(x) # (B, T, C)
        q = self.query(x) # (B, T, C)

        # compute attention scores
        wei = q @ k.transpose(-2, -1) * (C ** -0.5) # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T) mask out future tokens
        wei = F.softmax(wei, dim=-1) # (B, T, T) softmax to get attention weights
        wei = self.dropout(wei) # (B, T, T) apply dropout to attention weights

        # perform the weighted aggregation of the values
        v = self.value(x) # (B, T, C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

# multi-head self attention
class MultiHeadAttention(nn.Module):
    """multiple heads of self attention in parallel"""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) # create multiple heads
        self.proj = nn.Linear(n_embed, n_embed) # linear layer to project the concatenated head outputs back to embedding dimension
        self.dropout = nn.Dropout(dropout) # dropout layer for regularization
    
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # concatenate the outputs of all heads, shape (B, T, num_heads * head_size)
        out = self.dropout(self.proj(out)) # project back to embedding dimension and apply dropout
        return out

In [146]:
# transformer block p2: assemble all layers

class Block(nn.Module):
    """MoE transformwer block: communication followed by computation (multi-head self attention + SpraseMoe)"""

    def __init__(self, n_embed, n_head, num_experts, top_k):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.smoe = SparseMoE(n_embed, num_experts, top_k)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    
    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # multi-head attention with residual
        x = x + self.smoe(self.ln2(x)) # sparse MoE with residual
        return x

In [147]:
# define entire language model architecture
# w/ input + transofmer block + output 

class SparseMoeLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed) # embedding layer
        self.position_embedding_table = nn.Embedding(block_size, n_embed) # positional embedding layer
        self.blocks = nn.Sequential(*[Block(n_embed, n_head=n_head, num_experts=num_experts, top_k=top_k) for _ in range(n_layer)]) # stack of transformer blocks
        self.ln_f = nn.LayerNorm(n_embed) # final layer norm
        self.lm_head = nn.Linear(n_embed, vocab_size) # language model head

    def forward(self, idx, targets=None):
        B, T = idx.shape # batch size, context length

        tok_emb = self.token_embedding_table(idx) # (B, T, n_embed) token embeddings
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embed) positional embeddings
        x = tok_emb + pos_emb # (B,

        # idx and targets are both (B,T) tensor of integers
        token_emb = self.token_embedding_table(idx) # (B, T, n_embed) token embeddings
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device)) # (T, n_embed) positional embeddings
        x = token_emb + pos_emb # (B, T, n_embed) combine token and positional embeddings
        x = self.blocks(x) # (B, T, n_embed) pass through transformer blocks
        x = self.ln_f(x) # (B, T, n_embed) final layer norm
        logits = self.lm_head(x) # (B, T, vocab_size) compute logits for next token prediction

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C) # reshape for loss computation
            targets = targets.view(B*T) # reshape targets
            loss = F.cross_entropy(logits, targets) # compute cross-entropy loss

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # udx is the (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:] # crop context to block size
            # get the predictions
            logits, loss = self(idx_cond) # get logits for the current context
            # focus only on the last time step
            logits = logits[:, -1, :] # focus on the last time step
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # convert to probabilities
            # sample from distribution
            idx_next = torch.multinomial(probs, num_samples=1) # sample the next token
            # append sampled idx to running seq
            idx = torch.cat((idx, idx_next), dim=1) # append sampled token to the context
        return idx


In [148]:
# generate training and test data
torch.manual_seed(42)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# all unique chars in the text
chars = sorted(list(set(text)))
vocab_size = len(chars) # vocab size

# create mapping from chars to integers and vice versa
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# train and test splits
data = torch.tensor(encode(text), dtype=torch.long) # encode the entire text dataset as
n = int(0.9*len(data)) # first 90% for training, rest for testing
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs and targets
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,)) # random starting indices for the batch
    x = torch.stack([data[i:i+block_size] for i in ix]) #
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) # targets are the next tokens
    # return x, y
    # x, y = x.to(device), y.to(device)
    return x, y


In [149]:
# define loss function

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval() # set model to evaluation mode
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train() # set model back to training mode
    return out

In [150]:
# define training loop params and other hyperparams

import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn import init

# hpyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 32 # what is the maximum context length for predictions?
max_iters = 50000 # SET TO AS HIGH AS 50000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# if torch.cuda.is_available():
#     device = 'cuda'
# elif torch.backends.mps.is_available():
#     device = 'mps'
# else:
#     device = 'cpu'

# print(f"using device: {device}")
eval_iters = 400
head_size = 16
n_embed = 128
n_head = 8
n_layer = 8
dropout = 0.1
num_experts = 8
top_k = 2



In [151]:
# init entire model

def kaiming_init_weights(m):
    if isinstance(m, nn.Linear):
        init.kaiming_normal_(m.weight)

In [152]:
#

model = SparseMoeLanguageModel()
model.apply(kaiming_init_weights) # initialize weights with Kaiming initialization
# model.to(device) # move model to device (GPU or CPU)

SparseMoeLanguageModel(
  (token_embedding_table): Embedding(65, 128)
  (position_embedding_table): Embedding(32, 128)
  (blocks): Sequential(
    (0): Block(
      (sa): MultiHeadAttention(
        (heads): ModuleList(
          (0-7): 8 x Head(
            (key): Linear(in_features=128, out_features=16, bias=False)
            (query): Linear(in_features=128, out_features=16, bias=False)
            (value): Linear(in_features=128, out_features=16, bias=False)
            (dropout): Dropout(p=0.1, inplace=False)
          )
        )
        (proj): Linear(in_features=128, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (smoe): SparseMoE(
        (router): NoisyTopkRouter(
          (topkroute_linear): Linear(in_features=128, out_features=8, bias=True)
          (noise_linear): Linear(in_features=128, out_features=8, bias=True)
        )
        (experts): ModuleList(
          (0-7): 8 x Expert(
            (net): Sequential(
             

In [ ]:
# run pre-training loop

# not using MLflow
m = model.to(device)

# print num of params in model
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# create pytorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while eval loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # eval loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

8.996545 M parameters
step 0: train loss 5.1166, val loss 5.1317
step 100: train loss 2.6626, val loss 2.6705
step 200: train loss 2.4803, val loss 2.4954
step 300: train loss 2.4103, val loss 2.4223
step 400: train loss 2.3135, val loss 2.3307
step 500: train loss 2.2513, val loss 2.2679
step 600: train loss 2.1595, val loss 2.2106
step 700: train loss 2.0935, val loss 2.1528
step 800: train loss 2.0424, val loss 2.1066
step 900: train loss 2.0013, val loss 2.0711
step 1000: train loss 1.9662, val loss 2.0398
step 1100: train loss 1.9279, val loss 2.0308
step 1200: train loss 1.9063, val loss 1.9954
step 1300: train loss 1.8672, val loss 1.9771
step 1400: train loss 1.8376, val loss 1.9651
step 1500: train loss 1.8188, val loss 1.9338
step 1600: train loss 1.7884, val loss 1.9287
step 1700: train loss 1.7844, val loss 1.9135
step 1800: train loss 1.7520, val loss 1.9069
step 1900: train loss 1.7448, val loss 1.8910
step 2000: train loss 1.7303, val loss 1.8822
step 2100: train loss 1.

In [ ]:
# save model

torch.save(model.state_dict(), 'sparse_moe.pt')

In [ ]:
# inference
context = torch.zeros((1, 1), dtype=torch.long, device=device) # start with a single token (B=1, T=1)
print(decode(model.generate(context, max_new_tokens=2000)[0].tolist())) # generate 500 new tokens and decode to string


IO be yYhast ar g st thay hoanthe ard

Iastrr MER
ORGE braro noufsbles wind w nomouncf wane s ofist, w f chaswhp bcit nomears n shbl dhy.
O'non, I fircld sthounont'de thaing Cl'st I's feseltwh ighume, g,
Wh wand thase athang aththafar sed m beroulle wer,

ARI d hanador:
Adecitamthorerd s forrd n minimeas thutiverds;
Tougls b wit I s berallousast, sthive ber ngire my I thit aroulatt wis; I Istreme ofsal ican u,
Ife ffinen nelasif wefark owisth nthed ot sthat han ncasomperd ONI sur hatheef?

PING:

K:
DUARESn'loro MTACatow C'derthelllllot, ot bfon ik se boninghere honthoist iseratr ths he ine.
LOLwinousl by hithan fingl mge hathas-s-f thag I hon il'ss ghazist o birerst heemancot k brepre. : ghir anouth aulfof Oung fouso t thintind, ts
Tat benor thyou tathay ame henthe-aith ghasue 'd.
Nencot treinowat n achaspiCERWhond n.
TiteC w nird ff's w Hais Isthetorsssthel Whatou kingmen I th erim bor? Has Yomike biecla s beav cous kereP Whilthe,

IamENENG-H:
OC fit t j:
Modifimy, tu id hy me taner

In [ ]:
# actual inference

prompt = "To be or not"
context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
# shape: (1, len(prompt_tokens))

print(decode(model.generate(context, max_new_tokens=200)[0].tolist()))


To be or nothe gst avereneearn, hathenghel tith mig sive ti; hebevesith haick wtllam I beche arist bars linen, foulend wisenot W? lere n goupont ththecou imacour ners dcht nandars. hyin.
miofpr Gfnivimirs afingre
